<a href="https://colab.research.google.com/github/Rohan0603/Daemon/blob/master/finetune_qwen2.5_3b_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-tune Qwen2.5-3B-Instruct with Unsloth (SFT)

This notebook fine-tunes `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` using Supervised Fine-Tuning (SFT) with Unsloth on a free Colab T4 GPU.

**Setup:**
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Click **Connect**
3. Run all cells in order

**Based on:** [unsloth-buddy](https://github.com/TYH-labs/unsloth-buddy) skill

## Cell 1: Install Unsloth & Dependencies

In [1]:
%%capture
!pip install unsloth
# Restart runtime if prompted (Runtime → Restart runtime)

## Cell 2: Verify GPU & Imports

In [ ]:
import torch, json
assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

from unsloth import FastLanguageModel
import unsloth, trl, transformers, datasets

print(json.dumps({
    "gpu": gpu_name,
    "vram_gb": round(vram_gb, 1),
    "unsloth": unsloth.__version__,
    "trl": trl.__version__,
    "transformers": transformers.__version__,
    "datasets": datasets.__version__,
    "cuda": torch.version.cuda,
}))
print("GPU ready!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


## Cell 3: Load Model & Apply LoRA

Loads `unsloth/Qwen2.5-3B-Instruct-bnb-4bit` in 4-bit QLoRA (~2GB VRAM).

In [ ]:
max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print(f"Model loaded. Trainable params: {model.print_trainable_parameters()}")

## Cell 4: Prepare Dataset

Choose ONE of the options below:
- **Option A**: Use a sample HuggingFace dataset (quick demo)
- **Option B**: Load your own JSONL/CSV file
- **Option C**: Create a custom dataset inline

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION A: Sample HuggingFace dataset (recommended for first run)
# Uses a small instruction-following dataset for quick demo
# ══════════════════════════════════════════════════════════════════════════════

from datasets import load_dataset

# Load a small instruction dataset (5K rows, fast to train)
dataset = load_dataset("BAAI/Infinity-Instruct", split="train", streaming=True)
dataset = dataset.shuffle(seed=42).select(range(500))  # Use 500 samples for quick demo

# Format to Qwen2.5 chat template
def format_chat(example):
    # Infinity-Instruct has 'instruction' and 'output' columns
    instruction = example.get("instruction", example.get("query", ""))
    output = example.get("output", example.get("response", ""))
    if not instruction or not output:
        return None
    messages = [
        {"role": "user", "content": instruction},
        {"role": "assistant", "content": output},
    ]
    return {"messages": messages}

dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
dataset = dataset.filter(lambda x: x is not None and x.get("messages") is not None)

print(f"Dataset: {len(dataset)} samples")
print(f"Example: {dataset[0]}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION B: Upload your own dataset (uncomment and modify)
# Supports JSONL or CSV files
# ══════════════════════════════════════════════════════════════════════════════

# from google.colab import files
# uploaded = files.upload()  # Upload your .jsonl or .csv file
#
# import pandas as pd
# filename = list(uploaded.keys())[0]
#
# if filename.endswith('.jsonl'):
#     df = pd.read_json(filename, lines=True)
# else:
#     df = pd.read_csv(filename)
#
# print(f"Columns: {df.columns.tolist()}")
# print(df.head())
#
# # Adapt this to your column names:
# # If you have 'question'/'answer' columns:
# def format_my_data(example):
#     return {"messages": [
#         {"role": "user", "content": example["question"]},
#         {"role": "assistant", "content": example["answer"]},
#     ]}
#
# dataset = Dataset.from_pandas(df).map(format_my_data, remove_columns=df.columns.tolist())

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# OPTION C: Create a custom dataset inline
# Replace the examples below with your own data
# ══════════════════════════════════════════════════════════════════════════════

# from datasets import Dataset
#
# my_data = [
#     {"messages": [{"role": "user", "content": "What is Python?"},
#                    {"role": "assistant", "content": "Python is a high-level programming language..."}]},
#     {"messages": [{"role": "user", "content": "Explain loops."},
#                    {"role": "assistant", "content": "Loops repeat code..."}]},
#     # Add more examples here (aim for 100+ for real training)
# ]
#
# dataset = Dataset.from_list(my_data)

## Cell 5: Train with SFT

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,   # Effective batch size = 8
        max_steps = 200,                   # Increase for real training (e.g., 300-500)
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        warmup_steps = 10,
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Starting training...")
trainer_stats = trainer.train()
print(f"Training complete! Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## Cell 6: Save LoRA Adapters

In [ ]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("Adapters saved to lora_model/")

# List saved files
import os
for f in os.listdir("lora_model"):
    size = os.path.getsize(os.path.join("lora_model", f))
    print(f"  {f}: {size / 1e6:.1f} MB")

## Cell 7: Test Inference

In [ ]:
from unsloth import FastLanguageModel

# Reload for inference
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# Test with a prompt
messages = [
    {"role": "user", "content": "Explain what machine learning is in simple terms."},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

## Cell 8 (Optional): Export to GGUF

Export to GGUF format for use with Ollama, LM Studio, or llama.cpp.

In [ ]:
# Uncomment to export:
# model.save_pretrained_gguf("model_gguf", tokenizer, quantization_method="q4_k_m")
# print("GGUF exported!")
#
# Download the GGUF file:
# from google.colab import files
# import glob
# for f in glob.glob("model_gguf/*.gguf"):
#     files.download(f)

## Cell 9 (Optional): Push to Hugging Face Hub

In [ ]:
# Uncomment and set your HF token:
# HF_TOKEN = "hf_YOUR_TOKEN_HERE"
# model.push_to_hub("your-username/qwen2.5-3b-finetuned", token=HF_TOKEN)
# tokenizer.push_to_hub("your-username/qwen2.5-3b-finetuned", token=HF_TOKEN)

## Download Adapters

Download the `lora_model/` folder from the Colab file browser (left panel → folder icon → right-click → Download).